# Effect of background music type and time awake on typing performance



In [8]:
import pathlib, textwrap, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import sqlite3

DATA_PATH = pathlib.Path("Data/typing_tests.csv")
OUT_DIR = pathlib.Path("Output/")
ACCURACY_MIN = 0.50          # threshold to drop buggy rows
OUT_DIR.mkdir(exist_ok=True)
pio.kaleido.scope.default_format = "svg"      # vector graphics by default
pio.kaleido.scope.default_width  = 900
pio.kaleido.scope.default_height = 600
pio.templates.default = "plotly_white"
PLOT_WIDTH = 900
PLOT_HEIGHT = PLOT_WIDTH / 1.5

warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

In [9]:
def load_data(csv_path: pathlib.Path) -> pd.DataFrame:
    """Load CSV and ensure correct dtypes."""
    df = pd.read_csv(csv_path)
    df["music_type"] = df["music_type"].astype("category")
    df["time_awake"] = (
        df["time_awake"].replace({"<1h": "1h"}).astype("category")  # nicer labels
    )
    df.drop(columns=["timestamp"], inplace=True)
    return df


def clean_data(df: pd.DataFrame, acc_min: float = ACCURACY_MIN) -> pd.DataFrame:
    """Drop rows with obviously wrong accuracy (design bug)."""
    return df.loc[df["typing_accuracy"] >= acc_min].copy()

def remove_outliers_iqr(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Remove rows with outliers in any of the specified columns using Tukey's IQR method.
    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): List of column names to check for outliers.
    Returns:
        pd.DataFrame: DataFrame with outliers removed.
    """
    mask = pd.Series(True, index=df.index)
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask &= df[col].between(lower, upper)
    return df[mask].copy()
    



def save_to_latex_table(df: pd.DataFrame, fname: str, caption: str, label: str):
    """Save DataFrame as a standalone .tex file (booktabs)."""
    path = OUT_DIR / fname
    with path.open("w") as f:
        f.write(
            textwrap.dedent(
                rf"""
        \begin{{table}}[htbp]
            \centering
            \caption{{{caption}}}
            \label{{{label}}}
            \footnotesize
            {df.to_latex(index=True, escape=False, float_format="%.3f")}
        \end{{table}}
        """
            )
        )
    print(f"LaTeX table saved → {path}")



df_raw = load_data(DATA_PATH)
df = clean_data(df_raw, acc_min=ACCURACY_MIN)
df = remove_outliers_iqr(df, ["typing_accuracy", "typing_speed_wpm"])
df["music_type"] = df["music_type"].cat.reorder_categories(
    ["Silence", "Classical", "Hardstyle"], ordered=True
)
df["time_awake"] = df["time_awake"].cat.reorder_categories(
    ["1h", "4-8h", "12h+"], ordered=True
)
df.head()

,id,music_type,time_awake,typing_speed_wpm,typing_accuracy,participant_id
1,20,Classical,12h+,86.4,0.881944,3
2,21,Hardstyle,12h+,88.6,0.873589,3
4,23,Classical,12h+,40.4,0.980198,9
6,25,Classical,12h+,55.0,0.941818,12
8,27,Silence,12h+,66.4,0.927711,12


In [10]:
# ---------- 3. descriptive statistics ----------
desc = (df
        .groupby(["music_type","time_awake"], observed=False)
        .agg(WPM_mean=("typing_speed_wpm","mean"),
             WPM_sd  =("typing_speed_wpm","std"),
             ACC_mean=("typing_accuracy","mean"),
             ACC_sd  =("typing_accuracy","std"))
        .round(2))
save_to_latex_table(desc,
           "descriptive.tex",
           "Descriptive statistics (M ± SD) for each condition.",
           "tab:descr")

LaTeX table saved → Output/descriptive.tex


In [11]:
# ---------- 4. publication-quality figures ----------
def violin_plot(
    xAxis: str, yAxis: str, sortBy: str, xlabel: str, ylabel: str, func_name: str
):
    fig = px.violin(
        df,
        x=xAxis,
        y=yAxis,
        color=sortBy,
        box=True,
        points="all",
        labels={xAxis: xlabel, yAxis: ylabel},
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )
    fig.update_layout(
        title=f"Distribution of {ylabel}", violingap=0.3, legend_title_text=""
    )
    fig.write_image(OUT_DIR / f"{func_name}-{sortBy}.svg")
    fig.write_image(OUT_DIR / f"{func_name}-{sortBy}.png", scale=2)  # high-res bitmap
    return fig


violin_plot_wpm = violin_plot(
    xAxis="music_type",
    xlabel="Music Type",
    yAxis="typing_speed_wpm",
    ylabel="Typing speed (WPM)",
    sortBy="time_awake",
    func_name="violin_wpm",
)
violin_plot_wpm.show()

violin_plot_acc = violin_plot(
    xAxis="music_type",
    xlabel="Music Type",
    yAxis="typing_accuracy",
    ylabel="Typing accuracy",
    sortBy="time_awake",
    func_name="violin_acc",
)
violin_plot_acc.show()

violin_plot_wpm_music = violin_plot(
    xAxis="time_awake",
    xlabel="Time awake",
    yAxis="typing_speed_wpm",
    ylabel="Typing speed (WPM)",
    sortBy="music_type",
    func_name="violin_wpm_music",
)
violin_plot_wpm_music.show()

violin_plot_acc_music = violin_plot(
    xAxis="time_awake",
    xlabel="Time awake",
    yAxis="typing_accuracy",
    ylabel="Typing accuracy",
    sortBy="music_type",
    func_name="violin_acc_music",
)
violin_plot_acc_music.show()

In [12]:
def interaction_plot(metric: str, ylab: str, fname: str):
    means = (df.groupby(["music_type","time_awake"])[metric]
               .mean().reset_index())
    fig = px.line(means, x="time_awake", y=metric, color="music_type",
                  markers=True,
                  labels={"time_awake":"Time awake","music_type":"Music type",
                          metric: ylab})
    fig.update_layout(title=f"Interaction plot for {ylab}",
                      legend_title_text="")
    fig.write_image(OUT_DIR / f"{fname}.svg")
    fig.write_image(OUT_DIR / f"{fname}.png", scale=2)
    return fig

interatcion_plot_wpm = interaction_plot("typing_speed_wpm", "Typing speed (WPM)", "int_wpm")
interatcion_plot_acc = interaction_plot("typing_accuracy", "Typing accuracy", "int_acc")

interatcion_plot_wpm.show()
interatcion_plot_acc.show()

/tmp/ipykernel_276765/1381532381.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_276765/1381532381.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [13]:
# ---------- 5. inferential statistics ----------
# 5.1  Mixed-effects models (participant random intercept)
model_wpm = smf.mixedlm(
    "typing_speed_wpm ~ music_type * time_awake",
    df, groups=df["participant_id"]).fit(reml=False)
model_acc = smf.mixedlm(
    "typing_accuracy ~ music_type * time_awake",
    df, groups=df["participant_id"]).fit(reml=False)

print(model_wpm.summary())
print(model_acc.summary())

# Save coefficient tables to LaTeX
save_to_latex_table(model_wpm.summary().tables[1].round(3),
           "anova_wpm.tex",
           "Mixed-effects model for typing speed.",
           "tab:wpm")
save_to_latex_table(model_acc.summary().tables[1].round(3),
           "anova_acc.tex",
           "Mixed-effects model for typing accuracy.",
           "tab:acc")

                        Mixed Linear Model Regression Results
Model:                    MixedLM         Dependent Variable:         typing_speed_wpm
No. Observations:         83              Method:                     ML              
No. Groups:               12              Scale:                      14.8024         
Min. group size:          1               Log-Likelihood:             -259.8778       
Max. group size:          27              Converged:                  Yes             
Mean group size:          6.9                                                         
--------------------------------------------------------------------------------------
                                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------------
Intercept                                   54.679    7.086  7.716 0.000 40.790 68.567
music_type[T.Classical]                     -0.800    1.814 -0.441 0

/home/william/miniconda3/envs/stat/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning:

The MLE may be on the boundary of the parameter space.



In [14]:
# 5.2  Post-hoc (Tukey HSD on cell means; ignores within-subject for simplicity)
df["cell"] = df["music_type"].astype(str) + "_" + df["time_awake"].astype(str)
tukey_wpm = pairwise_tukeyhsd(df["typing_speed_wpm"], df["cell"])
tukey_acc = pairwise_tukeyhsd(df["typing_accuracy"], df["cell"])
print(tukey_wpm)
print(tukey_acc)

         Multiple Comparison of Means - Tukey HSD, FWER=0.05         
    group1         group2     meandiff p-adj   lower    upper  reject
---------------------------------------------------------------------
Classical_12h+   Classical_1h  12.7778 0.9018 -16.8094 42.3649  False
Classical_12h+ Classical_4-8h  20.8889 0.3819  -8.6983  50.476  False
Classical_12h+ Hardstyle_12h+     2.36    1.0 -26.3694 31.0894  False
Classical_12h+   Hardstyle_1h  13.6889 0.8619 -15.8983  43.276  False
Classical_12h+ Hardstyle_4-8h   21.525 0.3881  -9.1006 52.1506  False
Classical_12h+   Silence_12h+    -0.74    1.0 -29.4694 27.9894  False
Classical_12h+     Silence_1h  13.5778 0.8672 -16.0094 43.1649  False
Classical_12h+   Silence_4-8h  18.6857 0.6353 -13.2254 50.5969  False
  Classical_1h Classical_4-8h   8.1111 0.9959 -23.5189 39.7411  False
  Classical_1h Hardstyle_12h+ -10.4178 0.9754 -41.2469 20.4113  False
  Classical_1h   Hardstyle_1h   0.9111    1.0 -30.7189 32.5411  False
  Classical_1h Hards